In [45]:
!pip install langgraph langchain langchain-openai langchain-community duckduckgo-search

In [46]:
!pip install -U ddgs

In [47]:
!pip install -U langchain-groq

In [48]:
pip install python-dotenv

In [49]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [50]:
import os
from dotenv import load_dotenv
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun

# Load API key
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Search tool
search = DuckDuckGoSearchRun()

# Agent state
class AgentState(TypedDict):
    question: str
    analysis: str
    search_results: str
    final_answer: str


# Node 1 — Reasoning
def ai_assistant(state: AgentState):

    question = state["question"]

    prompt = f"""
You are an AI Research Assistant.

Analyze the user question and understand what information is needed.

Question:
{question}
"""

    response = llm.invoke(prompt)

    return {
        "analysis": response.content
    }


# Node 2 — Web Search
def search_tool(state: AgentState):

    question = state["question"]

    results = search.run(question)

    return {
        "search_results": results
    }


# Node 3 — Final Answer
def generate_answer(state: AgentState):

    question = state["question"]
    search_results = state["search_results"]

    prompt = f"""
You are an AI Research Assistant.

Use the research data below to answer the question clearly.

Question:
{question}

Research Data:
{search_results}

Provide a concise summarized answer.
"""

    response = llm.invoke(prompt)

    return {
        "final_answer": response.content
    }


# Build graph
builder = StateGraph(AgentState)

builder.add_node("assistant", ai_assistant)
builder.add_node("search", search_tool)
builder.add_node("answer", generate_answer)

builder.add_edge(START, "assistant")
builder.add_edge("assistant", "search")
builder.add_edge("search", "answer")
builder.add_edge("answer", END)

graph = builder.compile()


# Run agent
result = graph.invoke({
    "question": "Explain Agentic AI"
})

print("\nFinal Answer:\n")
print(result["final_answer"])


Final Answer:

Based on the research data, I can summarize Agentic AI as follows:

**What is Agentic AI?**
Agentic AI is a stage of AI that uses sophisticated reasoning and iterative planning to solve complex, multi-step problems in dynamic environments.

**Key Characteristics:**

1. Builds on generative AI (gen AI) techniques using large language models (LLMs).
2. Uses a four-step process for problem-solving.
3. Can function in dynamic environments.

**Key Differences:**

1. Agentic AI differs from traditional AI models and co-pilot systems in its ability to solve complex, multi-step problems.
2. Agentic AI is sometimes helpful and sometimes dangerous, depending on the situation.

**Why Should You Care?**
Understanding Agentic AI is crucial as it has the potential to revolutionize problem-solving in various domains, but its capabilities also raise concerns about its potential risks and dangers.
